# 03 - Assign Cluster Labels to ALL Trajectories  (Stage 3)

**Purpose.** Load the chosen k-means model and assign a `cluster_label` to every
trajectory by nearest-centroid lookup (`predict`, no iteration).

- `complete` and `partial` trajectories are labelled (for `partial`, the last
  known position was already used as the day-100 proxy in Stage 1).
- `early_loss` trajectories have no usable features -> `cluster_label = -1`.

If `config.GROUP_MAP` is set, a merged `cluster_group` column is also added
(raw clusters merged into named pathway groups); otherwise `cluster_group`
mirrors `cluster_label`.

**Input.** `data/features.parquet`, `data/kmeans_models/kmeans_k{BEST_K}.pkl`.
**Output.** `data/labeled_trajectories.parquet`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import config as C
import pipeline as P
print("project root:", C.PROJECT_ROOT)

## 3.1  Choose k and load the model

In [ ]:
BEST_K = 20          # <-- set after inspecting notebook 02 (papermill: -p BEST_K <k>)

In [ ]:
import pickle
with open(C.MODELS_DIR / f"kmeans_k{BEST_K}.pkl", "rb") as f:
    km = pickle.load(f)
print("loaded model with", km.n_clusters, "clusters")

## 3.2  Predict labels for every labellable trajectory

In [ ]:
features = pd.read_parquet(C.FEATURES_FILE)
labelable = features.status != "early_loss"
X_all = P.build_feature_matrix(features[labelable])
labels = np.full(len(features), -1, dtype=np.int32)
labels[labelable.to_numpy()] = km.predict(X_all).astype(np.int32)
features["cluster_label"] = labels
print(features.cluster_label.value_counts().sort_index().to_string())

## 3.2b  Merge into pathway groups (optional)

If `config.GROUP_MAP` is non-empty, merge the raw clusters into groups; the
result is stored in `cluster_group`. With an empty map, `cluster_group` simply
equals `cluster_label`.

In [ ]:
if C.GROUP_MAP:
    groups, raw2grp = P.apply_group_map(features.cluster_label.to_numpy(), C.GROUP_MAP, BEST_K)
    features["cluster_group"] = groups.astype(np.int32)
    print("raw cluster -> group:", raw2grp)
    print(features.loc[features.cluster_group >= 0, "cluster_group"]
          .value_counts().sort_index().to_string())
else:
    features["cluster_group"] = features["cluster_label"]
    print("GROUP_MAP empty -> cluster_group mirrors cluster_label")

## 3.3  Save labelled trajectories

In [ ]:
features.to_parquet(C.LABELED_FILE, index=False)
print("saved", features.shape, "->", C.LABELED_FILE)

## 3.4  Summary

In [ ]:
lbl = features[features.cluster_label >= 0]
print(f"Labelled {len(lbl):,} trajectories into {BEST_K} clusters "
      f"({(features.cluster_label==-1).sum():,} early_loss left unlabelled).")
print(f"Saved to {C.LABELED_FILE}")